# Your top 10 is a coinflip past 2000 teams

You are sitting in the public top 10 of a competition. How much does that actually predict about
where you land once the private leaderboard opens?

The answer depends heavily on how many teams are in the field, and the size where it starts to
matter is lower than you might expect: **around 2,000 teams** -- which happens to be the exact
size class every currently-open competition in this account's own portfolio (S6E8, Biohub, RSNA,
and Kaggriculture) sits in right now, before any of their private boards have resolved.

This notebook measures the relationship directly from every already-closed Kaggle competition on
record, using **Meta Kaggle's own `Teams.csv`**, which carries both `PublicLeaderboardRank` and
`PrivateLeaderboardRank` per team. **Copy & Edit this** and swap `MY_COMP_SLUG` near the bottom for
your own competition to see which size bucket you're in and what the historical base rate says
about your own public top 10.

In [ ]:
import glob, time
t0 = time.time()

def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"{pattern} not found under /kaggle/input -- is kaggle/meta-kaggle mounted?"
    return hits[0]

TEAMS_PATH = find("Teams.csv")
COMPS_PATH = find("Competitions.csv")
print(f"reading {TEAMS_PATH}\n reading {COMPS_PATH}")

## Load every team's public and private rank, and every competition's team count

In [ ]:
import pandas as pd
import numpy as np

REQUIRED_TEAMS = ["CompetitionId", "PublicLeaderboardRank", "PrivateLeaderboardRank"]
REQUIRED_COMPS = ["Id", "Slug", "TotalTeams"]

teams = pd.read_csv(TEAMS_PATH, usecols=REQUIRED_TEAMS)
comps = pd.read_csv(COMPS_PATH, usecols=REQUIRED_COMPS)
n_comps_in_teams = teams["CompetitionId"].nunique()
print(f"{len(teams):,} team rows across {n_comps_in_teams:,} competitions [{time.time()-t0:.0f}s]")

## Define "survival": does the public top 10 stay the private top 10?

Only competitions with both ranks populated for at least 10 teams can even answer this question --
a competition that never resolved a private leaderboard, or drew fewer than 10 real entrants, has
no top 10 to check.

In [ ]:
resolved = teams.dropna(subset=["PublicLeaderboardRank", "PrivateLeaderboardRank"])
n_comps_resolved = resolved["CompetitionId"].nunique()
print(f"{len(resolved):,} team rows have both ranks populated, across {n_comps_resolved:,} competitions")

def survival_rate(g):
    pub_top10 = set(g.loc[g["PublicLeaderboardRank"] <= 10].index)
    if len(pub_top10) < 10:
        return None
    priv_top10 = set(g.loc[g["PrivateLeaderboardRank"] <= 10].index)
    return len(pub_top10 & priv_top10) / len(pub_top10)

survival = resolved.groupby("CompetitionId", sort=False).apply(survival_rate, include_groups=False)
survival = survival.dropna()
print(f"{len(survival):,} competitions have a full public top-10 AND resolved private ranks "
      f"[{time.time()-t0:.0f}s]")

## Bucket by team count

In [ ]:
comps_indexed = comps.set_index("Id")
sizes = comps_indexed.loc[survival.index, "TotalTeams"]

BUCKETS = [(0, 200), (200, 500), (500, 1000), (1000, 2000), (2000, 5000), (5000, np.inf)]
LABELS = ["<200", "200-500", "500-1000", "1000-2000", "2000-5000", "5000+"]

rows = []
for (lo, hi), label in zip(BUCKETS, LABELS):
    mask = (sizes >= lo) & (sizes < hi)
    vals = survival.loc[mask.index[mask]]
    rows.append({
        "bucket": label, "n_competitions": len(vals),
        "mean_survival": round(float(vals.mean()), 3) if len(vals) else None,
        "median_survival": round(float(vals.median()), 3) if len(vals) else None,
    })

bucket_df = pd.DataFrame(rows)
print(bucket_df.to_string(index=False))
bucket_df.to_csv("leaderboard_shakeup_by_size.csv", index=False)

## The relationship is monotonic, and it is not small

| team count | competitions measured | mean top-10 survival |
|---|---:|---:|
| under 200 | 2,560 | 85.2% |
| 200-500 | 194 | 73.2% |
| 500-1,000 | 165 | 71.7% |
| 1,000-2,000 | 177 | 66.0% |
| 2,000-5,000 | 128 | 48.6% |
| 5,000+ | 9 | 37.8% |

Below 200 teams, 85 out of 100 public-top-10 teams are still in the private top 10. Past 2,000
teams -- pooling the 2,000-5,000 and 5,000+ buckets, weighted by competition count -- it drops to
**roughly 48%**, a coinflip. The decline is monotonic across all six buckets with no reversal
anywhere in the curve.

**One honest caveat before anything else: the 5,000+ bucket has only 9 competitions.** Its 37.8%
point estimate is real but should not be trusted to two significant figures the way the 2,000-5,000
bucket's 128-competition estimate can be. Kaggriculture, at 5,804 teams, currently falls in that
thin bucket -- its own historical base rate is the least certain number on this page, even though
the direction (worse than the 2,000-5,000 bucket, not better) is consistent with the trend.

## Check your own competition (this is the knob -- swap the slug)

In [ ]:
MY_COMP_SLUG = "kaggriculture"   # <-- swap for any competition slug on Kaggle

slug_to_id = comps.set_index("Slug")["Id"].to_dict()
comp_id = slug_to_id.get(MY_COMP_SLUG)
if comp_id is None:
    print(f"{MY_COMP_SLUG}: not found in this Meta Kaggle snapshot")
else:
    live_row = comps_indexed.loc[comp_id] if comp_id in comps_indexed.index else None
    total_teams = int(live_row["TotalTeams"]) if live_row is not None else None
    bucket = None
    if total_teams is not None:
        for (lo, hi), lbl in zip(BUCKETS, LABELS):
            if lo <= total_teams < hi:
                bucket = lbl
                break
    own_survival = survival.get(comp_id)
    print(f"{MY_COMP_SLUG}: {total_teams:,} teams -> size bucket [{bucket}]"
          if total_teams is not None else f"{MY_COMP_SLUG}: team count unavailable")
    expected = bucket_df.loc[bucket_df["bucket"] == bucket, "mean_survival"]
    if len(expected):
        print(f"historical base rate for this bucket: {float(expected.iloc[0])*100:.1f}% of the "
              f"public top 10 typically survives to the private top 10")
    print(f"this competition's own resolved history in this snapshot: "
          f"{own_survival if own_survival is not None else 'not resolved yet, or not previously run'}")

## What this does and does not mean

This is a description of a historical relationship, not a mechanism. It does not prove *why*
bigger fields shake up more -- plausible candidates include tighter score clustering with more
entrants, more decimal places needed to break ties, and a larger pool of teams close enough to the
line for private-set noise to flip the order, but this notebook does not test between them.

It also does not mean a specific team's public rank is meaningless above 2,000 entrants -- 48%
survival is still far above the ~0.1% a random team would have of landing in a random top 10. It
means the *margin of trust* in "I am currently public-rank N" should shrink as the field grows, and
by 2,000+ teams that margin has roughly halved relative to a sub-200-team field. If you are deciding
how much weight to put on your own current public standing versus your own held-out validation
score, this is the actual, measured size of that discount -- not a guess.